# 📝 Prompt Templates and Messages in LangChain

## Learning Objectives
In this notebook, you will learn:
1. **Basic Prompt Templates** - Build reusable `ChatPromptTemplate` prompts with variable placeholders
2. **Message Types** - Construct conversations manually using `SystemMessage`, `HumanMessage`, and `AIMessage`
3. **Dynamic History with `MessagesPlaceholder`** - Inject a variable-length conversation history into a prompt
4. **Few-Shot Prompting** - Teach a model a pattern using `FewShotChatMessagePromptTemplate`
5. **Prompt Composition** - Combine reusable prompt fragments with the `+` operator

## Prerequisites
- Familiarity with LangChain basics (see notebooks `01_core_concepts.ipynb`-`03_prompt_messages.ipynb` in this folder)
- An `OPENAI_API_KEY` set in a `.env` file at the project root
- Basic understanding of chat message roles (system / human / ai)

---
## 🔧 Setup

We load environment variables from `.env` and import the LangChain building blocks used
throughout this notebook: prompt template classes, message classes, and the `ChatOpenAI`
chat model.

In [ ]:
# ============================================================================
# SETUP: Imports and Environment
# ============================================================================
from dotenv import load_dotenv

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
    MessagesPlaceholder,
)
from langchain_openai import ChatOpenAI

load_dotenv()

print("✅ Environment loaded and LangChain imports ready!")

---
## 🧩 Part 1: Basic `ChatPromptTemplate` Usage

`ChatPromptTemplate` is the standard way to build reusable, parameterized prompts in
LangChain. It supports a single templated string or a full list of role-tagged messages
(`system`, `human`, `ai`), with `{variable}` placeholders filled in at call time via
`.format_messages()`.

### Key Concepts:
- **`from_template`**: Quick single-message template (defaults to a `HumanMessage`)
- **`from_messages`**: Build a multi-turn template from `(role, text)` tuples

In [ ]:
# ============================================================================
# DEMO_BASIC_TEMPLATES: Basic ChatPromptTemplate Usage
# ============================================================================
def demo_basic_templates():
    """Basic ChatPromptTemplate usage."""

    # Simple template
    simple = ChatPromptTemplate.from_template("Translate '{text}' to {language}")

    messages = simple.format_messages(text="Hello, world!", language="French")
    print("Simple template:")
    print(f"  {messages}")

    # Multi-message template
    multi = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a translator. Be concise."),
            ("human", "Translate '{text}' to {language}"),
        ]
    )

    messages = multi.format_messages(text="Good morning", language="Japanese")
    print("\nMulti-message template:")
    for msg in messages:
        print(f"  {type(msg).__name__}: {msg.content}")


demo_basic_templates()

---
## 💬 Part 2: Working with Message Types

Under the hood, every prompt resolves to a list of message objects. You can also build
that list by hand using `SystemMessage`, `HumanMessage`, and `AIMessage` directly — useful
when reconstructing a conversation (e.g., from stored chat history) rather than templating
a fresh one.

> **Note**: This cell calls `ChatOpenAI`, so it requires a valid `OPENAI_API_KEY`.

In [ ]:
# ============================================================================
# DEMO_MESSAGE_TYPES: Working with Different Message Types
# ============================================================================
def demo_message_types():
    """Working with different message types."""

    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # Build conversation manually
    messages = [
        SystemMessage(content="You are a math tutor. Be brief."),
        HumanMessage(content="What's 5 * 5?"),
        AIMessage(content="25"),
        HumanMessage(content="And if I add 10?"),
    ]

    response = model.invoke(messages)
    print(f"Conversation result: {response.content}")


demo_message_types()

---
## 🕘 Part 3: `MessagesPlaceholder` for Dynamic History

`MessagesPlaceholder` reserves a spot in a template for a *list* of messages supplied at
call time — the standard pattern for injecting conversation history into a prompt without
knowing in advance how many turns it will contain.

In [ ]:
# ============================================================================
# DEMO_MESSAGES_PLACEHOLDER: Dynamic Conversation History
# ============================================================================
def demo_messages_placeholder():
    """Use MessagesPlaceholder for dynamic conversation history."""

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a helpful assistant."),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{question}"),
        ]
    )

    # Simulate conversation history
    history = [
        HumanMessage(content="My name is Paulo"),
        AIMessage(content="Nice to meet you, Paulo!"),
    ]

    messages = prompt.format_messages(history=history, question="What's my name?")

    print("With history placeholder:")
    for msg in messages:
        print(f"  {type(msg).__name__}: {msg.content[:50]}...")

    # Execute
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    chain = prompt | model
    response = chain.invoke({"history": history, "question": "What's my name?"})
    print(f"\nResponse: {response.content}")


demo_messages_placeholder()

---
## 🎯 Part 4: Few-Shot Prompting

`FewShotChatMessagePromptTemplate` teaches the model a pattern by rendering a list of
example input/output pairs into the prompt using a shared `example_prompt` template, then
slots that block of examples into a larger final prompt alongside the real user question.

In [ ]:
# ============================================================================
# DEMO_FEW_SHOT: Few-Shot Prompting with Examples
# ============================================================================
def demo_few_shot():
    """Few-shot prompting with examples."""

    # Define examples
    examples = [
        {"word": "happy", "opposite": "sad"},
        {"word": "fast", "opposite": "slow"},
        {"word": "big", "opposite": "small"},
    ]

    # Template for each example
    example_prompt = ChatPromptTemplate.from_messages(
        [
            ("human", "What's the opposite of '{word}'?"),
            ("ai", "The opposite of '{word}' is '{opposite}'."),
        ]
    )

    # Few-shot wrapper
    few_shot = FewShotChatMessagePromptTemplate(
        example_prompt=example_prompt,
        examples=examples,
    )

    # Final prompt
    final_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You give the opposite of words. Follow the examples."),
            few_shot,
            ("human", "What's the opposite of '{word}'?"),
        ]
    )

    # Test
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    chain = final_prompt | model

    response = chain.invoke({"word": "bright"})
    print(f"Few-shot result: {response.content}")


demo_few_shot()

---
## 🧬 Part 5: Prompt Composition

`ChatPromptTemplate` objects can be combined with the `+` operator, letting you assemble a
final prompt out of smaller, reusable fragments (e.g., a persona block plus a task block)
instead of rewriting the whole template every time one part changes.

In [ ]:
# ============================================================================
# DEMO_PROMPT_COMPOSITION: Compose Prompts from Reusable Parts
# ============================================================================
def demo_prompt_composition():
    """Compose prompts from reusable parts."""

    # Reusable system prompt
    persona = ChatPromptTemplate.from_messages(
        [("system", "You are a {role}. Your tone is {tone}.")]
    )

    # Reusable task prompt
    task = ChatPromptTemplate.from_messages([("human", "{task}")])

    # Combine
    full_prompt = persona + task

    # Test different combinations
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    chain = full_prompt | model

    # As a pirate
    response = chain.invoke(
        {
            "role": "pirate captain",
            "tone": "adventurous",
            "task": "Tell me about your ship",
        }
    )
    print(f"Pirate: {response.content[:100]}...")

    # As a scientist
    response = chain.invoke(
        {
            "role": "scientist",
            "tone": "precise and academic",
            "task": "Explain photosynthesis",
        }
    )
    print(f"\nScientist: {response.content[:100]}...")


demo_prompt_composition()

---
## 📝 Summary

In this notebook, we learned:

### 1. Prompt Templates
- **`ChatPromptTemplate.from_template`**: Quick single-message templates
- **`ChatPromptTemplate.from_messages`**: Multi-turn templates from role/text tuples
- **`MessagesPlaceholder`**: Reserves a slot for a variable-length list of messages (e.g., chat history)

### 2. Messages and Composition
- **`SystemMessage` / `HumanMessage` / `AIMessage`**: Build a conversation manually
- **`FewShotChatMessagePromptTemplate`**: Teach a pattern via input/output examples
- Prompt templates can be **composed with `+`** to combine reusable fragments

### Functions Defined in This Notebook
- `demo_basic_templates()`
- `demo_message_types()`
- `demo_messages_placeholder()`
- `demo_few_shot()`
- `demo_prompt_composition()`

### Next Steps
- Continue to `05_output_parsers_demo.ipynb` to learn how to parse structured output from LLM responses